# Added value of LLAS over CLAS and BLAS: 48h morphological OFFs

Does using the full LLAS set improve the correlation between OFF properties and
bandpower measures, compared to the more restrictive CLAS or BLAS subsets? If so, the
LLAS-only OFFs (those too small or short for CLAS/BLAS) carry genuine signal, which
would validate the liberal detection criteria.

Source: the whole-recording (48h) morphological OFFs
(`cnpix_local_sleep.morphological.mua.files.get_full_offs_path`), assembled in memory.
The `state_mode` toggle offers two ways to obtain the NREM and Wake states:

- `condition_pooled` (matches the per-condition notebook): each 48h OFF is tagged with
  the core condition whose statistical hypnogram covers its `start_time`; OFFs outside
  all six windows are dropped. NREM pools `Early.BSL.NREM` + `Early.REC.NREM.Match` +
  `Early.REC.NREM` + `Late.REC.NREM`; Wake pools `Early.NOD.Wake` + `Late.NOD.Wake`.
- `direct_48h`: the whole-recording OFFs are subset directly by their detection `state`
  column, NREM = `state == "NREM"` and Wake = `state == "Wake"` (strict, so REM/IS/MA/
  Other are excluded), with no condition restriction.

The steps are:

1. Compute per-group Spearman correlations separately on the full LLAS set, the CLAS
   subset, and the BLAS subset.
2. Compare paired per-group rho values across subsets.
3. Run meta-analysis for each subset and compare pooled estimates.
4. Separately analyze LLAS-exclusive and CLAS-exclusive OFFs to test whether OFFs unique
   to each tier carry signal on their own.


In [ ]:
import pathlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pubplots as pp
import scipy.stats

from cnpix_local_sleep import sps_conf
from cnpix_local_sleep.morphological import correlation_stats
from cnpix_local_sleep.morphological.mua import files as mua_files
from cnpix_local_sleep.morphological.pipeline import aggregate_experiment_offs as agg
from cnpix_local_sleep.morphological.pipeline.add_bandpower_to_offs import add_bandpower_columns
from cnpix_local_sleep.morphological.pipeline.postprocess_offs import postprocess_offs_frame
from cnpix_local_sleep import off_tables

In [ ]:
save_plots = True

# Two ways to obtain NREM / Wake states from the 48h morphological OFFs:
#   "condition_pooled": tag each OFF with the covering core condition, then pool
#                       the four NREM conditions / two Wake conditions.
#   "direct_48h":       subset the whole recording directly by detection state.
state_mode = "direct_48h"  # or "direct_48h"

xy_pairs = [
    ("area", "total_bipolar_inst_log_delta"),
    #("area", "total_bipolar_inst_log_eta"),
    ("median_duration", "max_bipolar_inst_log_delta"),
    #("median_duration", "max_bipolar_inst_log_eta"),
    ("span", "max_bipolar_inst_log_delta"),
    #("span", "max_bipolar_inst_log_eta"),
    ("median_trace", "max_bipolar_inst_log_delta"),
    #("median_trace", "max_bipolar_inst_log_eta"),
]
condition_filters = ["NREM", "Wake"] # also valid: "all"
group_cols = ["subject", "probe", "structure"]

# Pools used in condition_pooled mode.
NREM_CONDITIONS = [
    "Early.BSL.NREM",
    "Early.REC.NREM.Match",
    "Early.REC.NREM",
    "Late.REC.NREM",
]
WAKE_CONDITIONS = ["Early.NOD.Wake", "Late.NOD.Wake"]

merge_keys = ["subject", "probe", "structure", "start_time", "end_time"]

OUTPUT_DIR = pathlib.Path("./outputs/static_added_value")
(OUTPUT_DIR / state_mode).mkdir(parents=True, exist_ok=True)

In [ ]:
def _collect_cortical_48h_whole():
    """All cortical whole-recording (48h) OFFs, state-labeled.

    Mirrors ``aggregate_experiment_offs._collect_all_offs_from_full`` MINUS the
    per-condition assignment and out-of-window drop, so OFFs from the entire
    recording are retained with their detection ``state`` column intact. Cortex
    only, postprocessed in memory (clade, span_rel2max, ...). Requires NFS.
    """
    spsl_cx = sps_conf.get_subject_probe_structure_list(
        method=mua_files.METHOD,
        exclude_thalamus=True,
        exclude_striatum=True,
        exclude_other=True,
    )
    frames = []
    for subject, probe, structure in spsl_cx:
        fpath = mua_files.get_full_offs_path(subject, probe, structure)
        if not fpath.exists():
            continue
        sps_offs = pd.read_parquet(fpath)
        if sps_offs.empty:
            continue
        sps_offs["subject"] = subject
        sps_offs["probe"] = probe
        sps_offs["structure"] = structure
        postprocess_offs_frame(sps_offs, structure)
        frames.append(sps_offs.dropna(axis=1, how="all"))
    offs = pd.concat(frames, ignore_index=True)
    offs["structure"] = offs["structure"].astype("category")
    return offs.loc[offs["clade"] == "Cx"].reset_index(drop=True)


def _build_offs():
    """Load + categorize + bandpower-attach the cortical OFF frame (~10 min)."""
    # Assemble the cortical OFF frame for the selected state_mode.
    if state_mode == "condition_pooled":
        cortical = agg._collect_cortical_48h()  # condition-subset, exclusions applied
    elif state_mode == "direct_48h":
        cortical = _collect_cortical_48h_whole()  # whole recording, state-labeled
    else:
        raise ValueError(
            f"state_mode must be 'condition_pooled' or 'direct_48h', got {state_mode!r}"
        )

    # LLAS = liberal superset; CLAS / BLAS are nested filters of it.
    llas = agg._apply_filters(cortical, off_tables.llas_filters)
    clas_keys = set(
        map(tuple, agg._apply_filters(llas, off_tables.clas_filters)[merge_keys].values)
    )
    blas_keys = set(
        map(tuple, agg._apply_filters(llas, off_tables.blas_filters)[merge_keys].values)
    )

    # Assign each LLAS row its most restrictive category.
    llas_tuples = list(map(tuple, llas[merge_keys].values))
    llas["category"] = pd.Categorical(
        [
            "BLAS" if t in blas_keys else "CLAS" if t in clas_keys else "LLAS"
            for t in llas_tuples
        ],
        categories=["LLAS", "CLAS", "BLAS"],
        ordered=True,
    )

    # Attach per-OFF bandpower stats (delta / eta) used as the y-variables.
    return add_bandpower_columns(llas).reset_index(drop=True)


# Cache the expensive (~10 min) OFF-load + bandpower attach to parquet, keyed by
# state_mode. Set force_recompute = True after changing the load/filter/bandpower
# logic (xy_pairs changes do not require recompute -- they're only used
# downstream). Categoricals (category, structure) round-trip through parquet.
CACHE_DIR = OUTPUT_DIR / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
offs_cache = CACHE_DIR / f"offs_{state_mode}.parquet"
force_recompute = False

if offs_cache.exists() and not force_recompute:
    offs = pd.read_parquet(offs_cache)
    print(f"Loaded {len(offs):,} OFFs from cache: {offs_cache}")
else:
    offs = _build_offs()
    offs.to_parquet(offs_cache)
    print(f"Computed and cached {len(offs):,} OFFs -> {offs_cache}")

print(f"  {offs['category'].value_counts().to_dict()}")

In [ ]:
def _filter_by_condition(offs, condition_filter):
    if condition_filter == "all":
        return offs
    if state_mode == "condition_pooled":
        if condition_filter == "NREM":
            return offs[offs["condition"].isin(NREM_CONDITIONS)]
        elif condition_filter == "Wake":
            return offs[offs["condition"].isin(WAKE_CONDITIONS)]
    elif state_mode == "direct_48h":
        if condition_filter == "NREM":
            return offs[offs["state"] == "NREM"]
        elif condition_filter == "Wake":
            return offs[offs["state"] == "Wake"]
    raise ValueError(
        f"condition_filter must be 'all', 'NREM', or 'Wake', got {condition_filter!r}"
    )


# Define nested subsets
subset_defs = {
    "LLAS": lambda df: df,  # all OFFs
    "CLAS": lambda df: df[df["category"].isin(["CLAS", "BLAS"])],
    "BLAS": lambda df: df[df["category"] == "BLAS"],
}

# Define exclusive subsets (OFFs unique to each tier)
exclusive_defs = {
    "LLAS-exclusive": lambda df: df[df["category"] == "LLAS"],
    "CLAS-exclusive": lambda df: df[df["category"] == "CLAS"],
}

## Compute per-group correlations for each nested subset

For each condition and x-y pair, compute Spearman rho per (subject, probe, structure)
group separately on LLAS (all OFFs), CLAS (CLAS and BLAS OFFs only), and BLAS (BLAS OFFs
only).


In [ ]:
all_gc = {}  # (cond, x_col, y_col, subset) -> group_corrs DataFrame

for cond in condition_filters:
    cond_offs = _filter_by_condition(offs, cond)
    for x_col, y_col in xy_pairs:
        for subset_name, subset_fn in subset_defs.items():
            sub = subset_fn(cond_offs)
            gc = correlation_stats.compute_group_correlations(
                sub, x_col, y_col, group_cols
            )
            all_gc[(cond, x_col, y_col, subset_name)] = gc

print(f"Computed {len(all_gc)} group-correlation tables")

## Paired per-group comparison

For each group, compare rho values across LLAS, CLAS, and BLAS subsets.
Wilcoxon signed-rank test on paired differences tests whether the subset
choice systematically shifts correlations.

In [ ]:
paired_records = []

for cond in condition_filters:
    for x_col, y_col in xy_pairs:
        gc_llas = all_gc[(cond, x_col, y_col, "LLAS")]
        gc_clas = all_gc[(cond, x_col, y_col, "CLAS")]
        gc_blas = all_gc[(cond, x_col, y_col, "BLAS")]

        # Merge on group_cols to get paired rho values
        merged = gc_llas[group_cols + ["rho", "n"]].rename(
            columns={"rho": "rho_LLAS", "n": "n_LLAS"}
        )
        for name, gc in [("CLAS", gc_clas), ("BLAS", gc_blas)]:
            merged = merged.merge(
                gc[group_cols + ["rho", "n"]].rename(
                    columns={"rho": f"rho_{name}", "n": f"n_{name}"}
                ),
                on=group_cols,
                how="inner",
            )

        merged["diff_LLAS_CLAS"] = merged["rho_LLAS"] - merged["rho_CLAS"]
        merged["diff_LLAS_BLAS"] = merged["rho_LLAS"] - merged["rho_BLAS"]
        merged["diff_CLAS_BLAS"] = merged["rho_CLAS"] - merged["rho_BLAS"]

        for comp in ["LLAS_CLAS", "LLAS_BLAS", "CLAS_BLAS"]:
            diffs = merged[f"diff_{comp}"].values
            if len(diffs) >= 6:
                stat, p = scipy.stats.wilcoxon(diffs)
            else:
                stat, p = np.nan, np.nan
            paired_records.append(
                dict(
                    condition=cond,
                    x_col=x_col,
                    y_col=y_col,
                    comparison=comp.replace("_", " vs "),
                    n_groups=len(diffs),
                    mean_diff=np.mean(diffs),
                    median_diff=np.median(diffs),
                    wilcoxon_stat=stat,
                    wilcoxon_p=p,
                )
            )

paired_df = pd.DataFrame(paired_records)
display(
    paired_df.style.format(
        {
            "mean_diff": "{:+.4f}",
            "median_diff": "{:+.4f}",
            "wilcoxon_stat": "{:.1f}",
            "wilcoxon_p": "{:.2e}",
        }
    ).set_caption("Paired per-group rho differences across subsets")
)

In [ ]:
def plot_slope_chart(all_gc, cond, x_col, y_col, group_cols):
    """Slope chart showing how rho changes from LLAS -> CLAS -> BLAS per group."""
    gc_llas = all_gc[(cond, x_col, y_col, "LLAS")]
    gc_clas = all_gc[(cond, x_col, y_col, "CLAS")]
    gc_blas = all_gc[(cond, x_col, y_col, "BLAS")]

    merged = gc_llas[group_cols + ["rho"]].rename(columns={"rho": "rho_LLAS"})
    merged = merged.merge(
        gc_clas[group_cols + ["rho"]].rename(columns={"rho": "rho_CLAS"}),
        on=group_cols,
        how="inner",
    )
    merged = merged.merge(
        gc_blas[group_cols + ["rho"]].rename(columns={"rho": "rho_BLAS"}),
        on=group_cols,
        how="inner",
    )

    fig, ax = plt.subplots(figsize=(4, 5), constrained_layout=True)
    x_pos = [0, 1, 2]
    for _, row in merged.iterrows():
        label = " / ".join(str(row[c]) for c in group_cols)
        rhos = [row["rho_LLAS"], row["rho_CLAS"], row["rho_BLAS"]]
        ax.plot(x_pos, rhos, "o-", markersize=4, alpha=0.6, label=label)

    ax.set_xticks(x_pos)
    ax.set_xticklabels(["LLAS\n(all)", "CLAS\n(subset)", "BLAS\n(subset)"])
    ax.set_ylabel("Spearman rho")
    ax.set_title(f"{x_col} vs {y_col}\ncondition={cond}", fontsize=9)
    ax.axhline(0, color="grey", linestyle="--", linewidth=0.8)
    return fig


with pp.destination("figma"):
    for cond in condition_filters:
        for x_col, y_col in xy_pairs:
            fig = plot_slope_chart(all_gc, cond, x_col, y_col, group_cols)
            if save_plots:
                fig.savefig(
                    OUTPUT_DIR / state_mode / f"slope_{cond}_{x_col}_vs_{y_col}.svg"
                )
            plt.show()

## Meta-analytic comparison across subsets

Run DerSimonian-Laird meta-analysis for each subset and compare pooled rho
estimates. Side-by-side forest plots show per-group and pooled estimates for
LLAS vs CLAS vs BLAS.

In [ ]:
all_meta = {}  # (cond, x_col, y_col, subset) -> meta dict

for key, gc in all_gc.items():
    if len(gc) >= 2:
        all_meta[key] = correlation_stats.meta_analyze_correlations(gc)

# Summary table comparing pooled rho across subsets
meta_records = []
for cond in condition_filters:
    for x_col, y_col in xy_pairs:
        for subset_name in subset_defs:
            key = (cond, x_col, y_col, subset_name)
            if key in all_meta:
                m = all_meta[key]
                gc = all_gc[key]
                meta_records.append(
                    dict(
                        condition=cond,
                        x_col=x_col,
                        y_col=y_col,
                        subset=subset_name,
                        pooled_rho=m["overall_rho"],
                        ci_lo=m["ci_lo"],
                        ci_hi=m["ci_hi"],
                        p_value=m["p_value"],
                        i_squared=m["i_squared"],
                        k=m["k"],
                        effect_label=m["effect_label"],
                        total_n=gc["n"].sum(),
                    )
                )

meta_df = pd.DataFrame(meta_records)
meta_df["CI"] = meta_df.apply(lambda r: f"[{r['ci_lo']:.3f}, {r['ci_hi']:.3f}]", axis=1)
display(
    meta_df[
        [
            "condition",
            "x_col",
            "y_col",
            "subset",
            "pooled_rho",
            "CI",
            "p_value",
            "i_squared",
            "k",
            "total_n",
            "effect_label",
        ]
    ]
    .style.format(
        {
            "pooled_rho": "{:.3f}",
            "p_value": "{:.2e}",
            "i_squared": "{:.1f}%",
        }
    )
    .set_caption("Meta-analytic pooled Spearman rho by subset (LLAS, CLAS, BLAS)")
)

In [ ]:
def plot_forest(
    group_corrs: pd.DataFrame,
    meta: dict,
    title: str = "",
    ax: plt.Axes | None = None,
    plot_for_poster: bool = False,
) -> plt.Axes:
    """Forest plot of per-group correlations with pooled estimate.

    Parameters
    ----------
    group_corrs
        Output of :func:`compute_group_correlations`.
    meta
        Output of :func:`meta_analyze_correlations`.
    title
        Plot title.
    ax
        Matplotlib axes to draw on.  Created if *None*.
    """
    id_cols = [c for c in ("subject", "probe", "structure") if c in group_corrs.columns]
    labels = [
        " / ".join(str(row[c]) for c in id_cols) for _, row in group_corrs.iterrows()
    ]

    k = len(group_corrs)
    if ax is None:
        fig_width = 2.0 if plot_for_poster else 4.0
        fig_height = max(4, 0.15 * (k + 2))
        _, ax = plt.subplots(figsize=(fig_width, fig_height), constrained_layout=True)

    y_positions = np.arange(k)[::-1]

    # Per-group CIs
    rhos = group_corrs["rho"].values
    ci_los = group_corrs["ci_lo"].values
    ci_his = group_corrs["ci_hi"].values
    ax.errorbar(
        rhos,
        y_positions,
        xerr=[rhos - ci_los, ci_his - rhos],
        fmt="o",
        color="steelblue",
        ecolor="steelblue",
        elinewidth=1.2,
        markersize=4,
        capsize=2,
    )

    # Overall meta-analytic estimate (diamond)
    y_overall = -1.5
    diamond_hw = 0.4  # half-width in y
    diamond_x = [
        meta["ci_lo"],
        meta["overall_rho"],
        meta["ci_hi"],
        meta["overall_rho"],
    ]
    diamond_y = [
        y_overall,
        y_overall + diamond_hw,
        y_overall,
        y_overall - diamond_hw,
    ]
    ax.fill(diamond_x, diamond_y, color="firebrick", alpha=0.7)

    # Reference line at rho = 0
    ax.axvline(0, color="grey", linestyle="--", linewidth=0.8)

    # Labels
    if plot_for_poster:
        ax.set_yticks([])
        ax.set_yticklabels([])
    else:
        ax.set_yticks(list(y_positions) + [y_overall])
        ax.set_yticklabels(labels + ["Overall"], fontsize=7)
        ax.set_xlabel("Spearman rho")
        if title:
            ax.set_title(title, fontsize=9)

        # Right-side annotations: rho [CI]
        for i, (rho, lo, hi, n) in enumerate(
            zip(rhos, ci_los, ci_his, group_corrs["n"].values)
        ):
            ax.text(
                1.0,
                y_positions[i],
                f" {rho:+.3f} [{lo:+.3f}, {hi:+.3f}]  N={n}",
                transform=ax.get_yaxis_transform(),
                va="center",
                fontsize=6,
                family="monospace",
            )
        ax.text(
            1.0,
            y_overall,
            f" {meta['overall_rho']:+.3f} [{meta['ci_lo']:+.3f}, {meta['ci_hi']:+.3f}]",
            transform=ax.get_yaxis_transform(),
            va="center",
            fontsize=6,
            fontweight="bold",
            family="monospace",
        )

    if meta["overall_rho"] < 0:
        ax.set_xlim(-1.0, 0.05)
        ax.set_xticks([-1.0, -0.5, 0.0])
    else:
        ax.set_xlim(-0.05, 1.0)
        ax.set_xticks([0.0, 0.5, 1.0])
    ax.set_ylim(y_overall - 1, y_positions[0] + 1)
    return ax

In [ ]:
with pp.destination("figma"):
    for cond in condition_filters:
        for x_col, y_col in xy_pairs:
            fig, axes = plt.subplots(1, 3, figsize=(14, 5), constrained_layout=True)
            for ax, subset_name in zip(axes, subset_defs):
                key = (cond, x_col, y_col, subset_name)
                gc = all_gc[key]
                meta = all_meta.get(key)
                if meta is None:
                    ax.set_title(f"{subset_name}\n(too few groups)")
                    continue
                correlation_stats.plot_forest(
                    gc,
                    meta,
                    title=f"{subset_name} (N={gc['n'].sum():,})",
                    ax=ax,
                )
            fig.suptitle(f"{x_col} vs {y_col}  |  condition={cond}", fontsize=10)
            if save_plots:
                fig.savefig(
                    OUTPUT_DIR
                    / state_mode
                    / f"forest_compare_{cond}_{x_col}_vs_{y_col}.svg"
                )
            plt.show()

## Exclusive-subset analysis

Do OFFs unique to each tier carry signal on their own?

- LLAS-exclusive: OFFs in LLAS but not in CLAS or BLAS (`category == "LLAS"`), the extra
  OFFs captured by the most liberal criteria.
- CLAS-exclusive: OFFs in CLAS but not in BLAS (`category == "CLAS"`), the intermediate
  OFFs that pass CLAS but not BLAS.


In [ ]:
excl_gc = {}  # (cond, x_col, y_col, excl_name) -> group_corrs
excl_meta = {}

for cond in condition_filters:
    cond_offs = _filter_by_condition(offs, cond)
    for x_col, y_col in xy_pairs:
        for excl_name, excl_fn in exclusive_defs.items():
            sub = excl_fn(cond_offs)
            gc = correlation_stats.compute_group_correlations(
                sub, x_col, y_col, group_cols
            )
            excl_gc[(cond, x_col, y_col, excl_name)] = gc
            if len(gc) >= 2:
                excl_meta[(cond, x_col, y_col, excl_name)] = (
                    correlation_stats.meta_analyze_correlations(gc)
                )

# Summary table for exclusive subsets
excl_records = []
for cond in condition_filters:
    for x_col, y_col in xy_pairs:
        for excl_name in exclusive_defs:
            key = (cond, x_col, y_col, excl_name)
            gc = excl_gc[key]
            m = excl_meta.get(key)
            if m is None:
                continue
            excl_records.append(
                dict(
                    condition=cond,
                    x_col=x_col,
                    y_col=y_col,
                    subset=excl_name,
                    pooled_rho=m["overall_rho"],
                    ci_lo=m["ci_lo"],
                    ci_hi=m["ci_hi"],
                    p_value=m["p_value"],
                    i_squared=m["i_squared"],
                    k=m["k"],
                    effect_label=m["effect_label"],
                    total_n=gc["n"].sum(),
                )
            )

excl_df = pd.DataFrame(excl_records)
excl_df["CI"] = excl_df.apply(lambda r: f"[{r['ci_lo']:.3f}, {r['ci_hi']:.3f}]", axis=1)
display(
    excl_df[
        [
            "condition",
            "x_col",
            "y_col",
            "subset",
            "pooled_rho",
            "CI",
            "p_value",
            "i_squared",
            "k",
            "total_n",
            "effect_label",
        ]
    ]
    .style.format(
        {
            "pooled_rho": "{:.3f}",
            "p_value": "{:.2e}",
            "i_squared": "{:.1f}%",
        }
    )
    .set_caption("Meta-analytic pooled Spearman rho for exclusive subsets")
)

In [ ]:
with pp.destination("figma"):
    for cond in condition_filters:
        for x_col, y_col in xy_pairs:
            fig, axes = plt.subplots(1, 2, figsize=(10, 5), constrained_layout=True)
            for ax, excl_name in zip(axes, exclusive_defs):
                key = (cond, x_col, y_col, excl_name)
                gc = excl_gc[key]
                meta = excl_meta.get(key)
                if meta is None:
                    ax.set_title(f"{excl_name}\n(too few groups)")
                    continue
                correlation_stats.plot_forest(
                    gc,
                    meta,
                    title=f"{excl_name} (N={gc['n'].sum():,})",
                    ax=ax,
                )
            fig.suptitle(f"{x_col} vs {y_col}  |  condition={cond}", fontsize=10)
            if save_plots:
                fig.savefig(
                    OUTPUT_DIR
                    / state_mode
                    / f"forest_excl_{cond}_{x_col}_vs_{y_col}.svg"
                )
            plt.show()

## Summary

Combined comparison of pooled Spearman rho across all subset types (nested and
exclusive) for each condition and x-y pair.

In [ ]:
# Combine nested and exclusive meta-analysis results
combined_df = pd.concat([meta_df, excl_df], ignore_index=True)

display(
    combined_df[
        [
            "condition",
            "x_col",
            "y_col",
            "subset",
            "pooled_rho",
            "CI",
            "p_value",
            "total_n",
            "effect_label",
        ]
    ]
    .style.format(
        {
            "pooled_rho": "{:.3f}",
            "p_value": "{:.2e}",
        }
    )
    .set_caption("All subsets: pooled Spearman rho comparison")
)

In [ ]:
# Bar chart: pooled rho by subset for each x-y pair (condition="all")
subset_order = ["LLAS", "CLAS", "BLAS", "LLAS-exclusive", "CLAS-exclusive"]
subset_colors = {
    "LLAS": "#4c72b0",
    "CLAS": "#dd8452",
    "BLAS": "#55a868",
    "LLAS-exclusive": "#8da0cb",
    "CLAS-exclusive": "#e5c494",
}
xy_pair_slice = slice(0, 1)

with pp.destination("figma"):
    for cond in condition_filters:
        cond_data = combined_df[combined_df["condition"] == cond]
        pair_labels = [f"{x}\nvs\n{y}" for x, y in xy_pairs[xy_pair_slice]]

        n_pairs = len(xy_pairs[xy_pair_slice])
        fig, ax = plt.subplots(figsize=(n_pairs * 1.2, 4), constrained_layout=True)
        n_subsets = len(subset_order)
        bar_width = 0.15
        x_base = np.arange(n_pairs)

        for j, subset_name in enumerate(subset_order):
            sub = cond_data[cond_data["subset"] == subset_name]
            rhos = []
            ci_lo_vals = []
            ci_hi_vals = []
            for x_col, y_col in xy_pairs[xy_pair_slice]:
                row = sub[(sub["x_col"] == x_col) & (sub["y_col"] == y_col)]
                if len(row) == 1:
                    rhos.append(row.iloc[0]["pooled_rho"])
                    ci_lo_vals.append(row.iloc[0]["ci_lo"])
                    ci_hi_vals.append(row.iloc[0]["ci_hi"])
                else:
                    rhos.append(np.nan)
                    ci_lo_vals.append(np.nan)
                    ci_hi_vals.append(np.nan)

            rhos = np.array(rhos)
            errs = np.array([rhos - ci_lo_vals, np.array(ci_hi_vals) - rhos])
            offset = (j - n_subsets / 2 + 0.5) * bar_width
            ax.bar(
                x_base + offset,
                rhos,
                bar_width,
                yerr=errs,
                label=subset_name,
                color=subset_colors[subset_name],
                capsize=3,
                alpha=0.85,
            )

        ax.set_xticks(x_base)
        ax.set_xticklabels(pair_labels, fontsize=8)
        ax.set_ylabel("Pooled Spearman rho")
        ax.set_title(f"Pooled rho by subset  |  condition={cond}")
        ax.axhline(0, color="grey", linestyle="--", linewidth=0.8)
        ax.legend(fontsize=8, loc="best")

        if save_plots:
            fig.savefig(OUTPUT_DIR / state_mode / f"bar_rho_by_subset_{cond}.svg")
        plt.show()

In [ ]:
# Summary forest plots side by side: one panel per subset, per-group rho with
# its CI plus the pooled diamond (uses the pasted local `plot_forest`).
#
# NB: no sharex -- plot_forest hard-sets xlim/xticks per axis by the sign of the
# pooled rho, so a shared x-axis would let panels fight over the limits. Each
# panel self-scales to ~full rho range, so same-signed panels stay comparable.

# Nested subsets live in all_gc/all_meta; exclusive in excl_gc/excl_meta.
subset_source = {
    "LLAS": all_gc,
    "CLAS": all_gc,
    "BLAS": all_gc,
    "LLAS-exclusive": excl_gc,
    "CLAS-exclusive": excl_gc,
}
meta_source = {
    "LLAS": all_meta,
    "CLAS": all_meta,
    "BLAS": all_meta,
    "LLAS-exclusive": excl_meta,
    "CLAS-exclusive": excl_meta,
}

with pp.destination("figma"):
    for cond in condition_filters:
        for x_col, y_col in xy_pairs[xy_pair_slice]:
            fig, axes = plt.subplots(
                1,
                len(subset_order),
                figsize=(len(subset_order) * 1.9, (len(subset_source["LLAS"]) + 2) * 0.2),
                constrained_layout=True,
            )
            for ax, subset_name in zip(axes, subset_order):
                key = (cond, x_col, y_col, subset_name)
                gc = subset_source[subset_name].get(key)
                meta = meta_source[subset_name].get(key)
                if gc is None or meta is None:
                    ax.set_title(f"{subset_name}\n(too few groups)", fontsize=9)
                    ax.axis("off")
                    continue
                plot_forest(
                    gc,
                    meta,
                    title=f"{subset_name} (N={gc['n'].sum():,})",
                    ax=ax,
                    plot_for_poster=True
                )

            fig.suptitle(f"{x_col} vs {y_col}  |  condition={cond}", fontsize=10)
            if save_plots:
                fig.savefig(
                    OUTPUT_DIR
                    / state_mode
                    / f"forest_summary_side_by_side_{cond}_{x_col}_vs_{y_col}.svg"
                )
            plt.show()